---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group XX

- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
from pathlib import Path

import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from eda import compare_train_test, describe_data, explore_data, verify_quality
from evaluation import compare_with_sklearn, cross_validate, scores
from naive_bayes import NaiveBayes
from nb_compare import SklearnCategoricalNB, exact_equivalence
from preprocessing import PreparationPipeline, construct_features, drop_duplicates_and_invalid, split_train_val

from decision_tree import DecisionTree
from evaluation import accuracy, roc_auc
from knn import KNN
from logreg import LogisticRegression
from preprocessing import (CATEGORICAL, TARGET, CategoricalEncoder, Clipper, Imputer, Scaler, feature_cols,
                           handle_imbalance, make_views, numeric_pipeline, prepare, tree_pipeline)
from submission import check_reproducible, make_submission
from svm import SVM, GaussianKernel, LinearKernel, PolynomialKernel, _stratified_subsample

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED       = 42
    DATA_DIR   = ROOT / 'T2-IF3170-2026-2027'
    OUTPUT_DIR = ROOT / 'output'

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
eda_describe = describe_data(train_df)

### Insights

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
eda_target_corr = explore_data(train_df, CFG.OUTPUT_DIR / 'eda')
eda_target_corr.head(10)

### Insights

### Train Versus Test

The test set is used here for diagnosis only. Nothing is fitted from it.

In [ ]:
eda_shift = compare_train_test(train_df, test_df)

### Insights

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
eda_quality_train = verify_quality(train_df, 'train')
eda_quality_test = verify_quality(test_df, 'test')

### Insights

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
train_set, val_set = split_train_val(train_df, val_size=0.2, seed=CFG.SEED)

print(f'Train: {train_set.shape}  Validation: {val_set.shape}')
pd.DataFrame({
    'train': train_set[TARGET].value_counts(normalize=True),
    'validation': val_set[TARGET].value_counts(normalize=True),
}).round(4)

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
selected_attributes = feature_cols(train_set)

print(f'{len(selected_attributes)} attributes: {selected_attributes}')
print(f'Categorical: {CATEGORICAL}')

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
missing_cols = [c for c in selected_attributes if train_set[c].isna().any()]
missing_table = pd.DataFrame({
    'train': train_set[missing_cols].isna().mean(),
    'validation': val_set[missing_cols].isna().mean(),
    'test': test_df[missing_cols].isna().mean(),
    'cancel rate if missing': [train_set.loc[train_set[c].isna(), TARGET].mean() for c in missing_cols],
    'cancel rate if present': [train_set.loc[train_set[c].notna(), TARGET].mean() for c in missing_cols],
}).round(4)

imputer = Imputer().fit(train_set)
imputed_val = imputer.transform(val_set)
print('Fill values from train:', {c: imputer.fill_[c] for c in missing_cols})
print('Missing cells in validation after imputation:', int(imputed_val[selected_attributes].isna().sum().sum()))
missing_table

### Duplicates

This part covers repeated records.

In [ ]:
train_clean, clean_report = drop_duplicates_and_invalid(train_set)

cross_split_copies = val_set.merge(train_clean[selected_attributes + [TARGET]].drop_duplicates(),
                                   on=selected_attributes + [TARGET], how='inner')
print(f'Validation rows identical to a training row: {len(cross_split_copies)}')
pd.Series(clean_report, name='train rows')

### Outliers

This part covers values far from the rest of the data.

In [ ]:
constructed_train = construct_features(train_clean)
imputed_train = Imputer().fit(constructed_train).transform(constructed_train)
clipper = Clipper().fit(imputed_train)

outlier_table = clipper.bounds_.T.set_axis(['lower (1%)', 'upper (99%)'], axis=1)
outlier_table['train rows clipped'] = [
    int((~imputed_train[c].between(lo, hi)).sum()) for c, (lo, hi) in outlier_table.iterrows()
]
display(outlier_table)

imputed_train.groupby(imputed_train['prior_cancellations'] > 0)[TARGET].agg(['size', 'mean'])

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
numeric_attributes = [c for c in selected_attributes if c not in CATEGORICAL]
checks = {
    'zero nights': train_set['weekend_nights'] + train_set['weekday_nights'] == 0,
    'zero guests': train_set['num_adults'].eq(0) & train_set['num_children'].eq(0),
    'zero adults': train_set['num_adults'].eq(0),
    'zero nightly rate': train_set['nightly_rate_idr'].eq(0),
    'negative value': (train_set[numeric_attributes] < 0).any(axis=1),
}

print('Target values:', sorted(train_set[TARGET].unique()))
print('Categories:', {c: sorted(train_set[c].dropna().unique()) for c in CATEGORICAL})
pd.DataFrame({
    'train rows': {name: int(mask.sum()) for name, mask in checks.items()},
    'cancel rate': {name: train_set.loc[mask, TARGET].mean() for name, mask in checks.items()},
}).round(4)

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
derived_attributes = ['total_nights', 'total_guests', 'nightly_rate_per_guest']

construction_table = constructed_train[derived_attributes].describe().T
construction_table['spearman with target'] = [
    constructed_train[c].corr(constructed_train[TARGET], method='spearman') for c in derived_attributes
]
construction_table.round(3)

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
unscaled_prep = PreparationPipeline([construct_features, Imputer(), Clipper(), CategoricalEncoder()]).fit(train_clean)
unscaled_train, _ = unscaled_prep.transform(train_clean)
unscaled_val, _ = unscaled_prep.transform(val_set)

standard = Scaler('standard').fit(unscaled_train)
minmax = Scaler('minmax').fit(unscaled_train)
pd.DataFrame({
    'mean before': unscaled_train.mean(),
    'std before': unscaled_train.std(ddof=0),
    'mean standard (val)': standard.transform(unscaled_val).mean(),
    'std standard (val)': standard.transform(unscaled_val).std(ddof=0),
    'min min-max (val)': minmax.transform(unscaled_val).min(),
    'max min-max (val)': minmax.transform(unscaled_val).max(),
}).round(3)

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
encoder = CategoricalEncoder().fit(train_clean)
encoded_sample = encoder.transform(val_set.head())

print('Categories per attribute (from train):', {c: len(v) for c, v in encoder.categories_.items()})
pd.concat([val_set.head()[CATEGORICAL], encoded_sample.filter(like='=')], axis=1)

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
mi_selector = numeric_pipeline(select_k=20).fit(train_clean).steps[-1]
pca_check = numeric_pipeline(pca=0.95).fit(train_clean).steps[-1].pca_

print(f'PCA keeps {pca_check.n_components_} of {pca_check.n_features_in_} columns for 95% of the variance')
display(pd.Series(np.cumsum(pca_check.explained_variance_ratio_), index=range(1, pca_check.n_components_ + 1),
                  name='cumulative explained variance').round(4))
mi_selector.scores_.round(4).to_frame('mutual information with target')

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
balance_X, balance_y = numeric_pipeline().fit_transform(train_clean)
_, oversampled_y = handle_imbalance(balance_X, balance_y, seed=CFG.SEED)

pd.DataFrame({
    'train': train_set[TARGET].value_counts(),
    'train after cleaning': train_clean[TARGET].value_counts(),
    'train after oversampling': pd.Series(oversampled_y).value_counts(),
    'validation': val_set[TARGET].value_counts(),
})

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
views = make_views(train_clean, val_set, test_df)
(X_train, y_train), (X_val, y_val), (X_test, _) = views['numeric']
(T_train, _), (T_val, _), (T_test, _) = views['tree']
tree_categorical = [T_train.columns.get_loc(c) for c in CATEGORICAL]

pd.DataFrame({
    'numeric view (KNN, LR, SVM)': [X_train.shape, X_val.shape, X_test.shape,
                                    int(X_train.isna().sum().sum()), sorted(set(X_train.dtypes.astype(str)))],
    'tree view (DTL, NB)': [T_train.shape, T_val.shape, T_test.shape,
                            int(T_train.isna().sum().sum()), sorted(set(T_train.dtypes.astype(str)))],
}, index=['train', 'validation', 'test', 'missing cells in train', 'dtypes'])

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
for name, pipe in [('numeric view', numeric_pipeline()), ('tree view', tree_pipeline())]:
    print(f"{name}: {' -> '.join(getattr(s, '__name__', type(s).__name__) for s in pipe.steps)}")

assert (X_train.index == T_train.index).all() and (X_val.index == T_val.index).all()
assert len(X_test) == len(T_test) == len(test_df)


def prepare_numeric(train_part, val_part):
    return prepare(train_part, val_part, 'numeric')

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
knn_grid = []
for k in [5, 15, 25, 41]:
    for weights in ['uniform', 'distance']:
        proba = KNN(k=k, weights=weights).fit(X_train, y_train).predict_proba(X_val)
        knn_grid.append({'k': k, 'weights': weights, **scores(y_val, proba)})
knn_grid = pd.DataFrame(knn_grid)
knn_grid.round(4)

In [ ]:
knn_best = knn_grid.loc[knn_grid['accuracy'].idxmax()]
knn_metrics = []
for metric, p in [('euclidean', 2), ('manhattan', 1), ('minkowski', 3)]:
    model = KNN(k=int(knn_best['k']), metric=metric, p=p, weights=knn_best['weights']).fit(X_train, y_train)
    knn_metrics.append({'metric': metric, 'p': p, **scores(y_val, model.predict_proba(X_val))})
knn_metrics = pd.DataFrame(knn_metrics)

best_metric = knn_metrics.loc[knn_metrics['accuracy'].idxmax()]
knn_params = {'k': int(knn_best['k']), 'metric': best_metric['metric'], 'p': int(best_metric['p']),
              'weights': knn_best['weights']}
print('Chosen:', knn_params)
knn_metrics.round(4)

In [ ]:
knn_model = KNN(**knn_params).fit(X_train, y_train)
knn_val_proba = knn_model.predict_proba(X_val)

knn_variants = {'default pipeline': scores(y_val, knn_val_proba)}
for name, pipe, oversample in [('top 20 features (mutual information)', numeric_pipeline(select_k=20), False),
                               ('PCA 95% variance', numeric_pipeline(pca=0.95), False),
                               ('random oversampling', numeric_pipeline(), True)]:
    pipe.fit(train_clean)
    X_a, y_a = pipe.transform(train_clean)
    X_b, y_b = pipe.transform(val_set)
    if oversample:
        X_a, y_a = handle_imbalance(X_a, y_a, seed=CFG.SEED)
    knn_variants[name] = scores(y_b, KNN(**knn_params).fit(X_a, y_a).predict_proba(X_b))
pd.DataFrame(knn_variants).T.round(4)

In [ ]:
knn_cv = cross_validate(lambda: KNN(**knn_params), prepare_numeric, train_df, k=5, seed=CFG.SEED)
knn_cv.round(4)

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
dtl_model = DecisionTree(categorical_features=tree_categorical).fit(T_train, y_train)
dtl_val_proba = dtl_model.predict_proba(T_val)

print(f'Nodes: {dtl_model.n_nodes_}  Depth: {dtl_model.depth_}')
pd.Series(scores(y_val, dtl_val_proba), name='validation')

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
lr_models, lr_val_proba, lr_table = {}, {}, []
for penalty in ['l1', 'l2', 'elasticnet']:
    model = LogisticRegression(penalty=penalty).fit(X_train, y_train)
    lr_models[penalty] = model
    lr_val_proba[penalty] = model.predict_proba(X_val)
    lr_table.append({'penalty': penalty, 'epochs': model.n_iter_, 'converged': model.converged_,
                     'zero weights': int((model.coef_ == 0).sum()), **scores(y_val, lr_val_proba[penalty])})
pd.DataFrame(lr_table).set_index('penalty').round(4)

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
svm_kernels = {'linear': LinearKernel(), 'polynomial': PolynomialKernel(), 'gaussian': GaussianKernel()}
svm_models, svm_val_proba, svm_table = {}, {}, []
for name, kernel in svm_kernels.items():
    model = SVM(kernel=kernel).fit(X_train, y_train)
    svm_models[name] = model
    svm_val_proba[name] = model.predict_proba(X_val)
    svm_table.append({'kernel': name, 'support vectors': len(model.support_), 'converged': model.converged_,
                      'SMO iterations': model.n_iter_, **scores(y_val, svm_val_proba[name])})
pd.DataFrame(svm_table).set_index('kernel').round(4)

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
nb_train, nb_val = split_train_val(train_df, seed=CFG.SEED)

def prepare_nb(train_part, val_part, steps=()):
    train_part, _ = drop_duplicates_and_invalid(train_part)
    pipeline = PreparationPipeline(list(steps)).fit(train_part)
    return (*pipeline.transform(train_part), *pipeline.transform(val_part))

nb_X_tr, nb_y_tr, nb_X_va, nb_y_va = prepare_nb(nb_train, nb_val)
nb_model = NaiveBayes(alpha=1.0, n_bins=20).fit(nb_X_tr, nb_y_tr)
nb_val_proba = nb_model.predict_proba(nb_X_va)

print(f'Train: {nb_X_tr.shape}  Validation: {nb_X_va.shape}')
print(f'Majority-class accuracy on validation: {1 - nb_y_va.mean():.4f}')
pd.Series(scores(nb_y_va, nb_val_proba), name='validation')

### Insights

### Cross-Validation and Feature Choice

The same stratified 5-fold split (seed 42) is used for every model, and preparation is refitted inside each fold. The comparison asks whether the derived columns of `construct_features` help Naive Bayes.

In [ ]:
nb_cv_raw = cross_validate(lambda: NaiveBayes(n_bins=20), prepare_nb, train_df)
nb_cv_derived = cross_validate(
    lambda: NaiveBayes(n_bins=20),
    lambda a, b: prepare_nb(a, b, steps=(construct_features,)),
    train_df,
)
pd.concat({'raw features': nb_cv_raw.loc['mean'], 'with derived features': nb_cv_derived.loc['mean']}, axis=1).T

### Insights

### Number of Bins

Numeric attributes with more than 10 distinct values are cut into quantile bins. The number of bins is chosen from training folds only.

In [ ]:
nb_bins_grid = [5, 10, 20, 30, 40]
nb_bins_cv = pd.DataFrame({
    b: cross_validate(lambda b=b: NaiveBayes(n_bins=b), prepare_nb, train_df).loc['mean']
    for b in nb_bins_grid
}).T
nb_bins_cv.index.name = 'n_bins'
nb_bins_cv

### Insights

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
knn_sklearn = KNeighborsClassifier(n_neighbors=knn_params['k'], weights=knn_params['weights'],
                                   metric=knn_params['metric'], p=knn_params['p'], algorithm='brute')
compare_with_sklearn({'KNN': (KNN(**knn_params), knn_sklearn)}, X_train, y_train, X_val, y_val).round(4)

In [ ]:
# Decision Tree: scratch C4.5 vs DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.001)
dtl_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                             encoded_missing_value=np.nan).fit(T_train[CATEGORICAL])


def encode_tree(T):
    T = T.copy()
    T[CATEGORICAL] = dtl_encoder.transform(T[CATEGORICAL])
    return T.astype(float)


dtl_sklearn = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.001, random_state=CFG.SEED)
dtl_sklearn.fit(encode_tree(T_train), y_train)
pd.DataFrame({'scratch': scores(y_val, dtl_val_proba),
              'sklearn': scores(y_val, dtl_sklearn.predict_proba(encode_tree(T_val))[:, 1])}).T.round(4)

In [ ]:
# Logistic Regression: alpha = 1 / (C n), so both minimize the same objective
lr_comparison = {}
for penalty, model in lr_models.items():
    ratio = {'l1': 1.0, 'l2': 0.0, 'elasticnet': model.l1_ratio}[penalty]
    sk = SkLogisticRegression(C=1 / (model.alpha * len(X_train)), l1_ratio=ratio, solver='saga',
                              max_iter=20000, tol=1e-10).fit(X_train, y_train)
    lr_comparison[(penalty, 'scratch')] = scores(y_val, lr_val_proba[penalty])
    lr_comparison[(penalty, 'sklearn')] = {**scores(y_val, sk.predict_proba(X_val)[:, 1]),
                                           'max abs coef diff': np.abs(sk.coef_[0] - model.coef_).max()}
pd.DataFrame(lr_comparison).T.round(4)

In [ ]:
# SVM: SVC trained on the same stratified subsample the scratch SVM draws with its seed
X_val_np = X_val.to_numpy()
sklearn_kernels = {'linear': dict(kernel='linear'),
                   'polynomial': dict(kernel='poly', degree=3, gamma='auto', coef0=1.0),
                   'gaussian': dict(kernel='rbf', gamma='auto')}
svm_comparison = {}
for name, params in sklearn_kernels.items():
    model = svm_models[name]
    rows, _ = _stratified_subsample(y_train, model.max_samples, np.random.default_rng(model.random_state))
    sk = SVC(C=model.C, tol=model.tol, **params).fit(X_train.to_numpy()[rows], y_train[rows])
    ours, ref = model.decision_function(X_val_np), sk.decision_function(X_val_np)
    svm_comparison[name] = {
        'support vectors (scratch)': len(model.support_), 'support vectors (sklearn)': len(sk.support_),
        'same sign': np.mean(np.sign(ours) == np.sign(ref)),
        'accuracy (scratch)': accuracy(y_val, ours > 0), 'accuracy (sklearn)': accuracy(y_val, ref > 0),
        'roc_auc (scratch)': roc_auc(y_val, ours), 'roc_auc (sklearn)': roc_auc(y_val, ref),
    }
pd.DataFrame(svm_comparison).T.round(4)

### Naive Bayes

The scikit-learn counterpart is `CategoricalNB` with `KBinsDiscretizer` and `OrdinalEncoder`, fitted on the training part only (`src/nb_compare.py`). Missing values are one extra category for scikit-learn, while the scratch model skips the attribute for that row. The first check gives both models the same bin codes, so only the counting, smoothing and normalization are compared.

In [ ]:
exact_equivalence(nb_X_tr, nb_y_tr, nb_X_va, NaiveBayes)

In [ ]:
compare_with_sklearn(
    {'Naive Bayes': (NaiveBayes(n_bins=20), SklearnCategoricalNB(n_bins=20))},
    nb_X_tr, nb_y_tr, nb_X_va, nb_y_va,
)

In [ ]:
nb_cv_sklearn = cross_validate(lambda: SklearnCategoricalNB(n_bins=20), prepare_nb, train_df)
pd.concat({'scratch': nb_cv_raw.loc['mean'], 'sklearn': nb_cv_sklearn.loc['mean']}, axis=1).T

### Insights

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
# Write your code here

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
submission_path = CFG.OUTPUT_DIR / 'submission.csv'
submission = make_submission(KNN(**knn_params), numeric_pipeline(), train_df, test_df, submission_path)

print(f'Rows: {len(submission)}  Share predicted canceled: {submission[TARGET].mean():.4f}')
print('Identical to output/submission_knn.csv:', check_reproducible(submission_path, CFG.OUTPUT_DIR / 'submission_knn.csv'))
submission.head()

In [ ]:
lr_best = max(lr_val_proba, key=lambda name: scores(y_val, lr_val_proba[name])['accuracy'])
svm_best = max(svm_val_proba, key=lambda name: scores(y_val, svm_val_proba[name])['accuracy'])
per_model = {
    'decision_tree': (DecisionTree(categorical_features=tree_categorical), tree_pipeline()),
    f'logreg_{lr_best}': (LogisticRegression(penalty=lr_best), numeric_pipeline()),
    f'svm_{svm_best}': (SVM(kernel=svm_kernels[svm_best]), numeric_pipeline()),
    'naive_bayes': (NaiveBayes(alpha=1.0, n_bins=20), PreparationPipeline([])),
}

model_submissions = {'knn': submission}
for name, (model, pipeline) in per_model.items():
    model_submissions[name] = make_submission(model, pipeline, train_df, test_df,
                                              CFG.OUTPUT_DIR / f'submission_{name}.csv')

pd.DataFrame({
    name: {'share predicted canceled': sub[TARGET].mean(),
           'same prediction as KNN': (sub[TARGET].to_numpy() == submission[TARGET].to_numpy()).mean()}
    for name, sub in model_submissions.items()
}).T.round(4)

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.